# Day 1: LLM Fundamentals
## Tokenization · Data Pipeline · Attention Mechanics

**Duration:** ~2 hours | **GPU Time:** ~15 min

### What you will build

| Step | Output |
|------|--------|
| Environment setup | Verified GPU environment + deterministic seeds |
| Tokenization | GPT-2 BPE tokenizer, vocabulary exploration |
| Data pipeline | WikiText-103 → flat token tensors |
| Sliding window dataset | `(x, y)` pairs ready for next-token prediction |
| Embeddings | Token → vector space intuition |
| Self-attention | Scaled dot-product + causal mask from scratch |

Every primitive here maps directly onto a component of **NanoLlama v3**, which you build end-to-end in Day 2.

## 1  Environment Setup

Modern LLM training requires:
- **TF32** — NVIDIA's 19-bit format for matmuls (same exponent range as FP32, less mantissa). `allow_tf32 = True` gives ~3× throughput on A100/T4 with negligible accuracy loss.
- **Deterministic seeds** — `manual_seed` + `cuda.manual_seed_all` ensures reproducible weight initialization and data shuffling across runs.

In [ ]:
import os
import math
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch: {torch.__version__}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_gpus = torch.cuda.device_count()
print(f'Device: {device} | GPUs: {n_gpus}')

torch.manual_seed(42)
if device == 'cuda':
    torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(n_gpus):
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem_gb:.1f} GB)')

## 2  Tokenization

Neural networks operate on numbers, not characters. **Tokenization** maps raw text to a sequence of integer IDs from a fixed vocabulary.

### Byte-Pair Encoding (BPE)

GPT-2 uses BPE — a data-driven compression algorithm:
1. Start with individual bytes as the vocabulary.
2. Repeatedly merge the most frequent adjacent pair into a new token.
3. Stop when the vocabulary reaches the target size (50,257 for GPT-2).

**Why BPE?**
- Common words like `" the"` get a single token → efficient.
- Rare words split into subwords → handles unseen vocabulary gracefully.
- No out-of-vocabulary problem — every byte sequence is encodable.

**Key property:** the tokenizer is a fixed lookup table, not learned during LLM training. NanoLlama reuses GPT-2's BPE tokenizer unchanged.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
vocab_size = tokenizer.vocab_size
print(f'Tokenizer: GPT-2 BPE | Vocab size: {vocab_size:,}')

examples = [
    'The transformer architecture changed everything.',
    'LLMs learn from next-token prediction.',
    'Rotary position embeddings encode relative distance.',
]

print()
for text in examples:
    ids  = tokenizer.encode(text)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f'Input : {text!r}')
    print(f'Tokens: {toks}')
    print(f'IDs   : {ids}')
    print(f'Count : {len(ids)} tokens')
    print()

### Subword behaviour

Notice the leading space: GPT-2 BPE treats `" transformer"` (space + word) as a single token distinct from `"transformer"`. This makes tokenization context-sensitive at word boundaries.

Numbers typically split digit-by-digit. Technical terms often split at morpheme boundaries (`un` + `believ` + `able`). Domain-specific words like `LoRA` may become 2–3 tokens.

In [ ]:
words = [
    'king', 'kingdom', 'unbelievable', 'tokenization',
    'anthropic', '2024', 'GPU', 'LoRA', 'SwiGLU', 'RMSNorm',
]

print(f'{"Word":<20} Subword tokens')
print('-' * 55)
for w in words:
    toks = tokenizer.convert_ids_to_tokens(tokenizer.encode(w))
    print(f'  {w:<20} {toks}')

## 3  Data Pipeline: WikiText-103

NanoLlama trains on **WikiText-103** — ~100M tokens of quality Wikipedia prose.

### Tokenization strategy

We concatenate the entire corpus into a single flat tensor of token IDs. This avoids wasting context on padding and maximises information density per training step.

| Split | Lines | Tokens (approx) |
|-------|-------|------------------|
| train | ~1.8M | ~100M |
| validation | ~3.8K | ~220K |

We process in 5,000-line chunks to avoid memory spikes when calling `tokenizer.encode` on very long strings.

In [ ]:
from datasets import load_dataset

print('Loading WikiText-103...')
raw_dataset = load_dataset('wikitext', 'wikitext-103-raw-v1')
print(f'  Train lines: {len(raw_dataset["train"]):,}')
print(f'  Val lines  : {len(raw_dataset["validation"]):,}')

# ── Tokenize train ────────────────────────────────────────
print('\nTokenizing train split...')
train_lines = [line for line in raw_dataset['train']['text'] if line.strip()]
train_ids   = []
chunk_size  = 5000

for i in range(0, len(train_lines), chunk_size):
    chunk = '\n'.join(train_lines[i : i + chunk_size])
    train_ids.extend(tokenizer.encode(chunk))
    if i % 50000 == 0:
        print(f'  {i:>7,} / {len(train_lines):,} lines...')

print(f'  Train tokens: {len(train_ids):,}')

# ── Tokenize val ─────────────────────────────────────────
print('\nTokenizing val split...')
val_lines = [line for line in raw_dataset['validation']['text'] if line.strip()]
val_ids   = tokenizer.encode('\n'.join(val_lines))
print(f'  Val tokens  : {len(val_ids):,}')

# ── To tensors ───────────────────────────────────────────
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data   = torch.tensor(val_ids,   dtype=torch.long)
print(f'\ntrain_data: {train_data.shape}  dtype={train_data.dtype}')
print(f'val_data  : {val_data.shape}  dtype={val_data.dtype}')

del raw_dataset, train_lines, val_lines, train_ids, val_ids
if device == 'cuda':
    torch.cuda.empty_cache()
print('\nIntermediate lists freed.')

## 4  Sliding Window Dataset

Language models are trained on **next-token prediction**: given tokens `x[0..T-1]`, predict `x[1..T]`.

A `SlidingWindowDataset` cuts the flat token tensor into overlapping `(input, target)` pairs:

```
tokens = [a, b, c, d, e, f, g, h]   block_size=4, stride=2

window 0:  x=[a,b,c,d]   y=[b,c,d,e]
window 1:  x=[c,d,e,f]   y=[d,e,f,g]
window 2:  x=[e,f,g,h]   y=[f,g,h,?]  ← needs block_size+1 tokens, dropped
```

**stride < block_size** → overlapping windows → more training samples from the same data, at the cost of correlation between adjacent batches. NanoLlama uses `block_size=1024, stride=512` (50% overlap).

**Why stride ≠ block_size?** The model must learn to handle any context position, not just clean non-overlapping chunks. Overlapping windows also double the effective dataset size cheaply.

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, tokens, block_size=1024, stride=512):
        self.tokens     = tokens
        self.block_size = block_size
        self.starts     = list(range(0, len(tokens) - block_size - 1, stride))

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = self.starts[idx]
        x = self.tokens[s     : s + self.block_size]
        y = self.tokens[s + 1 : s + self.block_size + 1]
        return x, y


block_size = 1024
train_ds   = SlidingWindowDataset(train_data, block_size=block_size, stride=512)
val_ds     = SlidingWindowDataset(val_data,   block_size=block_size, stride=512)

print(f'Train windows : {len(train_ds):,}')
print(f'Val windows   : {len(val_ds):,}')

# ── Inspect one sample ───────────────────────────────────
x0, y0 = train_ds[0]
print(f'\nSample [0]')
print(f'  x shape : {x0.shape}  dtype={x0.dtype}')
print(f'  y shape : {y0.shape}')
print(f'  x[:8]   : {x0[:8].tolist()}')
print(f'  y[:8]   : {y0[:8].tolist()}   <- y is x shifted right by 1')
print()
print(f'  x tokens: {tokenizer.convert_ids_to_tokens(x0[:8].tolist())}')
print(f'  y tokens: {tokenizer.convert_ids_to_tokens(y0[:8].tolist())}')

## 5  Token Embeddings

Token IDs are integers — they carry no semantic meaning. An **embedding table** maps each integer to a dense floating-point vector in $\mathbb{R}^{d}$.

$$e_i = E[\text{id}_i] \quad E \in \mathbb{R}^{V \times d}$$

- $V$ = vocabulary size (50,257)
- $d$ = embedding dimension (512 in NanoLlama v3)

The embedding table is the **single largest component** of NanoLlama: $50{,}257 \times 512 \approx 25.7$M parameters (≈50% of the model).

**Weight tying:** NanoLlama ties the embedding table to the output projection head. The same matrix $E$ is used both to look up input embeddings and to score output logits. This halves the parameter count for the largest tensor and regularises the output space.

**No positional embedding table** — NanoLlama uses RoPE instead, which injects position information directly into the attention computation (Day 2).

In [ ]:
n_embd = 512
emb    = nn.Embedding(vocab_size, n_embd)

param_count = vocab_size * n_embd
print(f'Embedding table : {vocab_size:,} x {n_embd} = {param_count:,} params')
print(f'Memory (FP32)   : {param_count * 4 / 1e6:.1f} MB')
print()

sentence = 'The transformer architecture'
ids      = torch.tensor([tokenizer.encode(sentence)])
toks     = tokenizer.convert_ids_to_tokens(ids[0].tolist())

with torch.no_grad():
    vecs = emb(ids)          # (1, T, 512)

print(f'Sentence : {sentence!r}')
print(f'Tokens   : {toks}')
print(f'IDs      : {ids[0].tolist()}')
print(f'vecs     : {vecs.shape}   (batch=1, seq={len(toks)}, dim={n_embd})')
print()

# Cosine similarity between randomly initialised vectors
# After training these cluster by semantics — near-zero now is expected
print('Cosine similarities (random init — near 0 is expected):')
for i in range(len(toks)):
    for j in range(i + 1, len(toks)):
        s = F.cosine_similarity(vecs[0, i:i+1], vecs[0, j:j+1], dim=-1).item()
        print(f'  {toks[i]!r} <-> {toks[j]!r}: {s:+.4f}')

print()
print('After training, tokens with related meanings cluster together in this space.')

## 6  Self-Attention

Attention allows every token to gather information from every other token (subject to masking). It is the mechanism that gives transformers their power.

### Scaled dot-product attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- **Q (Query):** what this token is looking for
- **K (Key):** what each token offers
- **V (Value):** what each token contributes if selected
- **$\sqrt{d_k}$ scale:** prevents dot products from growing large in magnitude as $d_k$ increases, which would push softmax into regions with near-zero gradients

### Causal masking

Language models are **autoregressive** — token $i$ must not attend to tokens $i+1, i+2, \ldots$ (future information).

We enforce this by adding $-\infty$ to future positions before softmax. After softmax, those positions become exactly 0 — the token cannot extract any information from them.

```
Mask (T=4):          After masking (row = query, col = key):
1 0 0 0             token 0 attends only to itself
1 1 0 0             token 1 attends to 0, 1
1 1 1 0             token 2 attends to 0, 1, 2
1 1 1 1             token 3 attends to all
```

NanoLlama precomputes this mask as a buffer inside `GQAttention` (Day 2).

In [ ]:
def scaled_dot_product_attention(Q, K, V, causal=True):
    """
    Q, K, V : (B, H, T, d_head)
    Returns  : output (B, H, T, d_head), weights (B, H, T, T)
    """
    B, H, T, d = Q.shape
    scale  = math.sqrt(d)
    scores = Q @ K.transpose(-2, -1) / scale   # (B, H, T, T)

    if causal:
        mask   = torch.tril(torch.ones(T, T, device=Q.device)).bool()
        scores = scores.masked_fill(~mask, float('-inf'))

    weights = torch.softmax(scores, dim=-1)
    out     = weights @ V
    return out, weights


# ── Demo: 4 tokens, 2 heads, 8-dim head ─────────────────
B, H, T, d = 1, 2, 4, 8
Q = torch.randn(B, H, T, d)
K = torch.randn(B, H, T, d)
V = torch.randn(B, H, T, d)

out, W = scaled_dot_product_attention(Q, K, V, causal=True)

print(f'Q, K, V : {Q.shape}   (B={B}, H={H}, T={T}, d_head={d})')
print(f'Output  : {out.shape}')
print(f'Weights : {W.shape}')
print()
print('Attention weights — head 0 (lower-triangular causal structure):')
print(W[0, 0].detach().numpy().round(3))
print()
print('Causal check — upper-triangle sum (must be 0.0):', W[0, 0].triu(1).sum().item())
print('Row sums (each row is a probability distribution over past tokens):')
print(W[0, 0].sum(dim=-1).detach().numpy().round(6))

### Multi-head attention

Running attention once with the full $d$ dimensions forces all heads to compete for the same representation. **Multi-head attention** runs $H$ independent attention heads in parallel, each with dimension $d_{head} = d / H$, then concatenates:

$$\text{MHA}(x) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\,W_o$$

Each head learns to attend to different aspects: one may track syntactic dependencies, another co-reference, another semantic similarity.

**Day 2 goes further:** NanoLlama uses **Grouped Query Attention (GQA)** — 8 query heads share only 4 key-value head pairs. This cuts KV-cache memory by 2× at inference with minimal quality loss.

In [ ]:
def multi_head_attention(x, W_q, W_k, W_v, W_o, n_head):
    """
    x    : (B, T, C)
    W_*  : nn.Linear(C, C)
    Returns: (B, T, C)
    """
    B, T, C = x.shape
    d_head  = C // n_head

    def project_and_split(W):
        return W(x).view(B, T, n_head, d_head).transpose(1, 2)

    Q = project_and_split(W_q)   # (B, H, T, d_head)
    K = project_and_split(W_k)
    V = project_and_split(W_v)

    out, _ = scaled_dot_product_attention(Q, K, V, causal=True)

    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return W_o(out)


C      = 64
n_head = 4
B, T   = 2, 6

W_q = nn.Linear(C, C, bias=False)
W_k = nn.Linear(C, C, bias=False)
W_v = nn.Linear(C, C, bias=False)
W_o = nn.Linear(C, C, bias=False)

x   = torch.randn(B, T, C)
y   = multi_head_attention(x, W_q, W_k, W_v, W_o, n_head)

print(f'Input  : {x.shape}   (B={B}, T={T}, C={C})')
print(f'Output : {y.shape}   same shape — each token enriched with context')
print(f'Heads  : {n_head} x {C // n_head}-dim')

## 7  Day 1 Summary

| Concept | What you built | NanoLlama v3 counterpart |
|---------|---------------|-------------------------|
| Tokenization | GPT-2 BPE, 50,257 vocab | `AutoTokenizer.from_pretrained('gpt2')` |
| Data | WikiText-103 → `train_data`, `val_data` tensors | Same tensors used in training loop |
| Dataset | `SlidingWindowDataset(tokens, 1024, 512)` | Exact class used in Day 2 |
| Embeddings | `nn.Embedding(vocab_size, n_embd)` | `self.tok_emb` in `NanoLlama` |
| Attention | Scaled dot-product + causal mask | Core of `GQAttention` |
| Multi-head | Project → split heads → attend → concat | `GQAttention` with GQA twist |

### Day 2 preview: NanoLlama v3 architecture upgrades

| GPT-2 component | NanoLlama v3 replacement | Why |
|-----------------|--------------------------|-----|
| Learned position embedding | **RoPE** | Encodes relative distance directly in Q/K; generalises to longer contexts |
| LayerNorm | **RMSNorm** | Removes mean subtraction; ~10% faster; no bias |
| GELU MLP | **SwiGLU** | Gated FFN with better gradient flow; state-of-the-art loss-per-param |
| Full MHA (8Q/8KV) | **GQA (8Q/4KV)** | Half the KV-cache; same quality at this scale |
| Bias everywhere | **No bias** | Regularisation; matches Llama/Mistral convention |